In [4]:
import numpy as np
import pandas as pd

from lidar_object_tracking.dataset import load_json_data
from lidar_object_tracking.config import PROCESSED_DATA_DIR
from lidar_object_tracking.plots import plot_with_slider

In [5]:
dataset = load_json_data(data_directory = PROCESSED_DATA_DIR,scene_num= '004')

dataset[50].shape

(1250, 3)

In [6]:
plot_with_slider(dataset)

In [7]:
from sklearn.cluster import HDBSCAN

hdb = HDBSCAN(copy = False)

clustering_by_frame = {}
for idx in range(80):
    clustering_by_frame[idx] = hdb.fit(dataset[idx])

In [8]:

print(dataset[50].shape)
print(hdb.fit(dataset[50]).labels_.shape)

clustering_by_frame[50].labels_.shape

(1250, 3)
(1250,)


(1250,)

In [9]:
for idx in range(80):
    print(idx)
    dataset[idx]['label'] = clustering_by_frame[idx].labels_

dataset[0].head()

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79


,xr,yr,z,label
42941,-7.122000,9.278491,0.511409,4
42942,-7.168666,9.321409,0.340035,4
43064,-6.960327,9.206873,0.526868,4
43127,-6.952079,9.228537,0.138343,4
43188,-6.902435,9.228531,0.141479,4


In [ ]:
for label in 


,xr,yr,z,label
49077,-2.907587,9.307557,0.113303,0
49129,-2.884511,9.336241,0.107746,0
49183,-2.852196,9.338909,0.107460,0
49235,-2.808936,9.337504,0.266802,0
49289,-2.785563,9.366006,0.261561,0


In [ ]:
frame = dataset[0].groupby('label')[['xr','yr','z']].mean()

frame.loc[2]

In [ ]:
center_of_mass_dict = {}

for idx in range(len(dataset)):
    centroid_by_label = {}
    cluster_centers = dataset[idx].groupby('label')[['xr','yr','z']].mean()
    for label in cluster_centers.index:
        if label >-1:
            centroid_by_label[label] = list(cluster_centers.loc[label][['xr','yr','z']])
    center_of_mass_dict[idx] = centroid_by_label

center_of_mass_dict
    

In [ ]:

def euclid_dist(vector1, vector2):
    num_coord = len(vector1)
    sum_squares = np.sum([(vector1[coord] - vector2[coord])**2 for coord in range(num_coord)])
    return sum_squares**.5

vector1 = [3,0,1]
vector2 = [0,4,3]

euclid_dist(vector1,vector2)

In [ ]:
vector1 = center_of_mass_dict[0][0]

distances = np.zeros(len(center_of_mass_dict[1]))
print(distances)
for key in center_of_mass_dict[1].keys():
    dist = euclid_dist(vector1, center_of_mass_dict[1][key])
    distances[key] = dist

print(np.argmin(distances))


In [ ]:
track_clusters = {}
for key in center_of_mass_dict[0].keys():
    vector1 = center_of_mass_dict[0][key]
    track_cluster_by_frame = [key]
    for idx in range(1,len(center_of_mass_dict)):
        distances = np.zeros(len(center_of_mass_dict[idx]))
        for label in center_of_mass_dict[idx].keys():
            dist = euclid_dist(vector1, center_of_mass_dict[idx][label])
            distances[label] = dist
        next_label = int(np.argmin(distances))
        vector1 = center_of_mass_dict[idx][next_label]
        track_cluster_by_frame.append(next_label)

    track_clusters[track_cluster_by_frame[0]] = track_cluster_by_frame

In [ ]:
track_clusters

In [ ]:
dataset[0][dataset[0]['label'] == 1]

In [ ]:
dataset0 = {}
cluster_id = 1

cluster_tracker = track_clusters[cluster_id]

for i in range(len(dataset)):
    print(cluster_tracker[i])
    dataset0[i] = dataset[i][dataset[i]['label'] == cluster_tracker[i]]
    print(dataset0[i])

In [ ]:
plot_with_slider(dataset0)